# 01 | Data understanding and preprocessing

**Motive:** Check whether source data needs cleaning and define reproducible inputs for forecasting.

**Logic:** Inspect missing values, duplicates, CPU validity, timestamp continuity, VM coverage and metadata availability before preparing observations and constructing targets.

Code cells retain the computed tables and verification assertions. Findings and interpretations are recorded beside the corresponding evidence in the [final report](../reports/report.md).


In [1]:
from pathlib import Path
import logging
import sys
import pandas as pd
from IPython.display import Markdown, display
from cwfc.io import verify_checksums, load_readings, load_metadata
from cwfc.quality import run_checks, prepare_readings, observation_summary

ROOT = Path.cwd()
if not (ROOT / "challenge_data").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "challenge_data").is_dir(), "Run from the repository or notebooks directory"
DATA = ROOT / "challenge_data"
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
pd.set_option("display.max_colwidth", 90)


## 1. Verify the files and load the raw observations

Checksums detect changed or corrupted release files. Loading preserves the raw records so quality problems remain visible. Relative timestamps stay in seconds; CPU values use float64, and identifiers use categorical dtypes.


In [2]:
checksums = verify_checksums(DATA)
readings = load_readings(DATA)
metadata = load_metadata(DATA)
report = run_checks(readings, metadata)
facts = report["checks"].set_index("check")["value"]
file_checks = pd.DataFrame({"File": list(checksums), "Checksum result": ["PASS"] * len(checksums)})
display(file_checks)


,File,Checksum result
0,vm_cpu_readings.csv.gz,PASS
1,vm_metadata.csv.gz,PASS
2,DATA_CARD.md,PASS


## 2. Checking missing values

Check **every column in each file**. Zero means that column has no missing values. Each dataset has its own table, so fields that belong only to the other file do not produce confusing blank or NaN cells.


In [3]:
for label, frame in [("CPU readings", readings), ("VM metadata", metadata)]:
    display(Markdown(f"### {label}"))
    missing = frame.isna().sum()
    table = pd.DataFrame({
        "Column": frame.columns,
        "Missing values": missing.to_numpy(),
        "Missing (%)": (missing / len(frame) * 100).round(2).to_numpy(),
    })
    display(table.style.hide(axis="index").format({"Missing values": "{:,}", "Missing (%)": "{:.2f}"}))


### CPU readings

Column,Missing values,Missing (%)
timestamp,0,0.00
vm_id,0,0.00
min_cpu,0,0.00
max_cpu,0,0.00
avg_cpu,0,0.00


### VM metadata

Column,Missing values,Missing (%)
vm_id,0,0.00
subscription_id,0,0.00
deployment_id,0,0.00
timestamp_vm_created,0,0.00
timestamp_vm_deleted,0,0.00
max_cpu,0,0.00
avg_cpu,0,0.00
p95_max_cpu,0,0.00
vm_category,0,0.00
virtual_core_bucket,0,0.00


## 3. Checking duplicated, invalid or irregular observations

A duplicate can give one VM too much weight in an aggregate. Invalid CPU values can distort averages, while missing timestamps can break the five-minute time grid. Matching VM populations also matters before metadata is joined in later analysis.


In [4]:
check_specs = [
    ("Duplicate VM/timestamp rows", "duplicate_vm_timestamp_rows", 0),
    ("CPU rows outside 0–100%", "cpu_out_of_range_rows", 0),
    ("Rows violating min ≤ average ≤ max", "cpu_order_violations", 0),
    ("Missing grid timestamps", "missing_grid_timestamps", 0),
    ("All timestamps aligned to the five-minute grid", "timestamp_grid_aligned", True),
    ("All consecutive timestamp differences equal 300 seconds", "all_timestamp_diffs_equal_cadence", True),
    ("VMs with incomplete valid coverage", "partial_coverage_vms", 0),
    ("Duplicate VM rows in metadata", "metadata_duplicate_vm_rows", 0),
    ("Metadata and readings contain the same VMs", "vm_populations_match", True),
    ("Deletion before creation", "invalid_lifetime_order_rows", 0),
]
quality_table = pd.DataFrame([
    {"Check": label, "Observed": facts[key], "Result": "PASS" if facts[key] == expected else "REVIEW"}
    for label, key, expected in check_specs
])
display(quality_table.style.hide(axis="index"))


Check,Observed,Result
Duplicate VM/timestamp rows,0,PASS
CPU rows outside 0–100%,0,PASS
Rows violating min ≤ average ≤ max,0,PASS
Missing grid timestamps,0,PASS
All timestamps aligned to the five-minute grid,True,PASS
All consecutive timestamp differences equal 300 seconds,True,PASS
VMs with incomplete valid coverage,0,PASS
Duplicate VM rows in metadata,0,PASS
Metadata and readings contain the same VMs,True,PASS
Deletion before creation,0,PASS


## 4. Prepare readings and audit row changes

The pipeline averages duplicate `(vm_id, timestamp)` rows, excludes incomplete CPU readings and rejects invalid CPU values or missing keys. It never fills individual VM readings. The audit then uses a complete 300-second grid: a slot with no observations has **zero observed VMs and NaN CPU**.


In [5]:
prepared = prepare_readings(readings)
observed = observation_summary(readings)
preparation_table = pd.DataFrame([
    ("Raw reading rows", f"{len(readings):,}"),
    ("Prepared reading rows", f"{len(prepared):,}"),
    ("Rows removed or combined", f"{len(readings) - len(prepared):,}"),
    ("Grid slots with no valid observations", f"{int(observed.no_observations.sum()):,}"),
    ("VM-level values imputed", "0"),
], columns=["Measure", "Result"])
display(preparation_table.style.hide(axis="index"))
assert not prepared.duplicated(["vm_id", "timestamp"]).any()
assert observed.loc[observed.no_observations, "aggregate_cpu_utilization"].isna().all()


INFO: Excluded 0 incomplete CPU readings (no imputation)


INFO: Averaging 0 duplicate VM/timestamp rows


INFO: Excluded 0 incomplete CPU readings (no imputation)


INFO: Averaging 0 duplicate VM/timestamp rows


Measure,Result
Raw reading rows,"1,340,000"
Prepared reading rows,"1,340,000"
Rows removed or combined,0
Grid slots with no valid observations,0
VM-level values imputed,0


## 5. Verify the release contract

Compare row counts, VM coverage, relative timestamp bounds and cadence against the supplied dataset description to detect incomplete or mismatched input files.


In [6]:
span_seconds = int(readings.timestamp.max() - readings.timestamp.min())
coverage = report["coverage"]
context_table = pd.DataFrame([
    ("Reading rows", f"{len(readings):,}"),
    ("VMs represented", f"{int(facts['readings_vm_count']):,}"),
    ("Distinct timestamps", f"{int(facts['distinct_timestamps']):,}"),
    ("First / last relative timestamp", f"{int(facts['first_timestamp_seconds']):,} / {int(facts['last_timestamp_seconds']):,} seconds"),
    ("First-to-last observation span", f"{span_seconds // 3600} hours {(span_seconds % 3600) // 60} minutes"),
    ("Sampling interval", f"{int(facts['cadence_seconds'])} seconds"),
    ("VMs with complete valid coverage", f"{coverage.coverage_fraction.eq(1).sum():,} ({coverage.coverage_fraction.eq(1).mean():.0%})"),
    ("Observed VM count: minimum / maximum", f"{observed.active_vm_count.min():,} / {observed.active_vm_count.max():,}"),
], columns=["Measure", "Verified value"])
display(context_table.style.hide(axis="index"))


Measure,Verified value
Reading rows,"1,340,000"
VMs represented,"10,000"
Distinct timestamps,134
First / last relative timestamp,"0 / 39,900 seconds"
First-to-last observation span,11 hours 5 minutes
Sampling interval,300 seconds
VMs with complete valid coverage,"10,000 (100%)"
Observed VM count: minimum / maximum,"10,000 / 10,000"


## 6. Inspect lifetime timestamps

**Motive:** Check the relationship between metadata event times and the observation window.

**Logic:** Count creation timestamps at relative zero, deletions after the window, deletions within six hours after the window and deletions within the first 24 trace hours. Summarize the recorded timestamp and duration distributions. Keep these calculations separate from forecast inputs.


In [7]:
lifetimes = report["lifetimes"]
last_timestamp = int(readings.timestamp.max())
lifetime_table = pd.DataFrame([('VMs with creation recorded at relative zero', f'{lifetimes.creation_at_trace_zero.sum():,}'), ('VMs with deletion after the observation window', f'{lifetimes.deletion_after_window.sum():,}'), ('VMs deleted within six hours after the window', f'{lifetimes.deletion_within_6h_after_window.sum():,}'), ('VMs deleted by relative day one', f'{lifetimes.timestamp_vm_deleted.le(86400).sum():,}'), ('Earliest deletion', f'{int(lifetimes.timestamp_vm_deleted.min()):,} seconds'), ('VMs whose deletion is recorded at 2,591,400 seconds', f'{lifetimes.timestamp_vm_deleted.eq(2591400).sum():,}')], columns=['Finding', 'Observed'])
display(lifetime_table.style.hide(axis="index"))
display(report["lifetime_distribution"].rename(columns={
    "timestamp_vm_created": "Recorded creation (seconds)",
    "timestamp_vm_deleted": "Recorded deletion (seconds)",
    "duration_seconds": "Recorded duration (seconds)",
}).style.format("{:,.1f}"))


Finding,Observed
VMs with creation recorded at relative zero,"10,000"
VMs with deletion after the observation window,"10,000"
VMs deleted within six hours after the window,37
VMs deleted by relative day one,122
Earliest deletion,"40,500 seconds"
"VMs whose deletion is recorded at 2,591,400 seconds","8,213"


,Recorded creation (seconds),Recorded deletion (seconds),Recorded duration (seconds)
count,"10,000.0","10,000.0","10,000.0"
mean,0.0,"2,289,625.6","2,289,625.6"
std,0.0,"708,100.8","708,100.8"
min,0.0,"40,500.0","40,500.0"
25%,0.0,"2,591,400.0","2,591,400.0"
50%,0.0,"2,591,400.0","2,591,400.0"
75%,0.0,"2,591,400.0","2,591,400.0"
max,0.0,"2,591,400.0","2,591,400.0"


## 7. Restrict inputs to information available at forecast time

**Motive:** Prevent future metadata or readings from entering model inputs.

**Logic:** Use only readings available at the forecast origin. Join only VM IDs and category/resource labels for descriptive segmentation. Exclude lifetime CPU summaries, deletion timestamps and derived lifetimes from forecast inputs; do not use them to fill readings. Preserve relative seconds rather than inventing calendar features.

## 8. Exercise preprocessing with imperfect input

Use a synthetic example with duplicate VM/timestamp rows, a missing timestamp and partial VM coverage. Run the same preprocessing functions and assert equal VM weighting, explicit gap flags and preserved missing CPU values.


In [8]:
smoke = pd.DataFrame([
    [0, "a", 0., 40., 20.], [0, "a", 10., 50., 30.],
    [0, "b", 0., 20., 10.], [600, "a", 0., 30., 15.],
], columns=["timestamp", "vm_id", "min_cpu", "max_cpu", "avg_cpu"])
smoke_summary = observation_summary(smoke)
display(smoke_summary.rename(columns={
    "timestamp": "Relative seconds", "active_vm_count": "Observed VMs",
    "aggregate_cpu_utilization": "Mean CPU (%)", "no_observations": "No valid observations",
    "missing_grid_timestamp": "Timestamp absent in source",
}).style.hide(axis="index").format({"Mean CPU (%)": "{:.1f}"}, na_rep="Missing (not imputed)"))
assert smoke_summary.active_vm_count.tolist() == [2, 0, 1]
assert smoke_summary.aggregate_cpu_utilization.iloc[0] == 17.5
assert pd.isna(smoke_summary.aggregate_cpu_utilization.iloc[1])


INFO: Excluded 0 incomplete CPU readings (no imputation)


INFO: Averaging 1 duplicate VM/timestamp rows


Relative seconds,Observed VMs,Mean CPU (%),No valid observations,Timestamp absent in source
0,2,17.5,False,False
300,0,Missing (not imputed),True,True
600,1,15.0,False,False


## 9. Construct the full target series

Build one row per five-minute timestamp **before splitting**. Each VM contributes once after the existing preparation policy. Optional category means join only VM IDs and workload categories from metadata.

| Target | Definition | Unit |
|---|---|---|
| `active_vm_count` | Distinct VMs with valid observations | VM count |
| `aggregate_cpu_utilization` | Mean observed VM `avg_cpu`, clipped to [0,100] | Percent |
| `p95_vm_cpu` | 95th percentile of observed VM `avg_cpu`, using linear interpolation | Percent |
| `hot_vm_share` | Fraction of observed VMs with `max_cpu > 80` | Fraction, 0–1 |
| `aggregate_peak_cpu` | Mean observed VM `max_cpu` | Percent |
| `segment_cpu_utilization::<category>` | Mean observed VM `avg_cpu` within the category | Percent |

A slot with no valid observations keeps count zero and missing CPU/share values. Flags distinguish absent source timestamps from timestamps whose readings were incomplete. No lifetime CPU metadata or deletion fields contribute to any target.


In [9]:
from cwfc.targets import build_targets, split_train_holdout

full_targets = build_targets(readings, metadata)
train_targets, holdout_targets = split_train_holdout(full_targets, n_train=110, horizon=24, min_train=48)

construction_checks = pd.DataFrame([
    ("One row per full-grid timestamp", len(full_targets) == len(observed)),
    ("Sorted and unique timestamps", full_targets.timestamp.is_monotonic_increasing and full_targets.timestamp.is_unique),
    ("Required targets match the observation audit", full_targets[["active_vm_count", "aggregate_cpu_utilization"]].equals(observed[["active_vm_count", "aggregate_cpu_utilization"]])),
    ("Training ends before holdout starts", train_targets.timestamp.max() < holdout_targets.timestamp.min()),
    ("No timestamp overlap", set(train_targets.timestamp).isdisjoint(holdout_targets.timestamp)),
], columns=["Check", "Passed"])
display(construction_checks.style.hide(axis="index"))
assert construction_checks.Passed.all()


INFO: Excluded 0 incomplete CPU readings (no imputation)


INFO: Averaging 0 duplicate VM/timestamp rows


Check,Passed
One row per full-grid timestamp,True
Sorted and unique timestamps,True
Required targets match the observation audit,True
Training ends before holdout starts,True
No timestamp overlap,True


### Chronological split and holdout boundary

Reserve the final 24 rows as the holdout and use the preceding 110 rows for training on the supplied release. The table displays **timestamps and row counts only** for the holdout; target values are not inspected for model selection.

On replacement data, the helper reserves the final `horizon` rows, uses up to `n_train` immediately preceding rows and requires at least `min_train` training rows. `n_train=None` uses all preceding history. Missing target rows are retained, so gaps cannot shift the forecast boundary.


In [10]:
split_table = pd.DataFrame([
    ("Training", len(train_targets), int(train_targets.timestamp.iloc[0]), int(train_targets.timestamp.iloc[-1]), "Available for later fitting and time-aware validation"),
    ("Holdout", len(holdout_targets), int(holdout_targets.timestamp.iloc[0]), int(holdout_targets.timestamp.iloc[-1]), "Reserved; no target summaries or evaluation here"),
], columns=["Window", "Rows", "First timestamp (seconds)", "Last timestamp (seconds)", "Use"])
display(split_table.style.hide(axis="index"))


Window,Rows,First timestamp (seconds),Last timestamp (seconds),Use
Training,110,0,32700,Available for later fitting and time-aware validation
Holdout,24,33000,39900,Reserved; no target summaries or evaluation here


### Training target summary

The statistics below use **only the training window**. P95 is computed across VMs at each timestamp; the row below then summarizes that time series. Hot-VM share is a fraction: 0.01 means 1% of observed VMs.


In [11]:
target_columns = [c for c in train_targets if c not in {"timestamp", "no_observations", "missing_grid_timestamp"}]
training_summary = train_targets[target_columns].agg(["min", "mean", "max"]).T
training_summary.columns = ["Minimum", "Mean", "Maximum"]
training_summary.index.name = "Target (training only)"
display(training_summary.style.format("{:,.4f}"))


,Minimum,Mean,Maximum
Target (training only),,,
active_vm_count,"10,000.0000","10,000.0000","10,000.0000"
aggregate_cpu_utilization,9.1116,9.4141,9.9122
p95_vm_cpu,34.2264,37.5470,41.1702
hot_vm_share,0.0251,0.0295,0.0450
aggregate_peak_cpu,17.6449,18.4414,19.9698
segment_cpu_utilization::Delay-insensitive,7.9001,8.2573,8.8776
segment_cpu_utilization::Interactive,10.9510,11.5051,12.0167
segment_cpu_utilization::Unknown,9.1655,10.3547,11.5607


### Verify the target pipeline on imperfect data

Reuse the synthetic duplicate/gap example. Both required and additional targets must preserve the missing 300-second slot. A small chronological split must reserve the last timestamp without dropping the gap.


In [12]:
smoke_targets = build_targets(smoke)
smoke_train, smoke_holdout = split_train_holdout(smoke_targets, horizon=1, min_train=2)
display(smoke_targets[["timestamp", "active_vm_count", "aggregate_cpu_utilization", "p95_vm_cpu", "hot_vm_share", "aggregate_peak_cpu"]].style.hide(axis="index").format(na_rep="Missing (not imputed)"))
assert smoke_targets.active_vm_count.tolist() == [2, 0, 1]
assert pd.isna(smoke_targets.loc[1, "aggregate_cpu_utilization"])
assert smoke_train.timestamp.tolist() == [0, 300]
assert smoke_holdout.timestamp.tolist() == [600]


INFO: Excluded 0 incomplete CPU readings (no imputation)


INFO: Averaging 1 duplicate VM/timestamp rows


timestamp,active_vm_count,aggregate_cpu_utilization,p95_vm_cpu,hot_vm_share,aggregate_peak_cpu
0,2,17.500000,24.250000,0.000000,32.500000
300,0,Missing (not imputed),Missing (not imputed),Missing (not imputed),Missing (not imputed)
600,1,15.000000,15.000000,0.000000,30.000000
